# Tariff

A **tariff** is a versioned list of cost-formula groups. Each version is active from its `start` timestamp until the next one begins.

Within a version, formulas are organised by **cost group**:

| Key | `CostGroup` | Description |
|---|---|---|
| `consumption` | `CONSUMPTION` | Cost per MWh of energy consumed |
| `injection` | `INJECTION` | Revenue/cost per MWh fed back into the grid |
| `capacity` | `CAPACITY` | Cost per MW of peak demand |
| `fixed` | `FIXED` | Recurring charge (daily, monthly, …) |

Each cost group holds one or more **named formulas** (see [formula.ipynb](./formula.ipynb)). A bare formula dict is shorthand for `{total: <formula>}`.

The tariff file is a YAML list; each entry needs a `start` timestamp and one or more cost-group keys.

## Setup

Register the `spot` price index and create a shared date range with sample meter readings for use throughout the notebook.

In [1]:
import datetime as dt

import pandas as pd
from helpers import display_yaml  # ty: ignore[unresolved-import]

from energy_cost.index import CSVIndex, Index
from energy_cost.meter import CostGroup, Meter, TimeseriesFrame
from energy_cost.tariff import Tariff

# Spot price index — monthly CSV covering 2025-06 to 2026-03
Index.register("spot", CSVIndex("../examples/indexes/foo.csv"))

# Shared query window: December 2025
start = dt.datetime(2025, 12, 1, tzinfo=dt.UTC)
end   = dt.datetime(2026,  1, 1, tzinfo=dt.UTC)
res   = dt.timedelta(hours=1)

# Sample meter readings (15-min, 1 kW load and 0.4 kW solar)
timestamps = pd.date_range(start, end, freq="15min", inclusive="left")
consumption = Meter(power=TimeseriesFrame({"timestamp": timestamps, "value": 0.25}))
injection   = Meter(power=TimeseriesFrame({"timestamp": timestamps, "value": 0.10}))

## 1. Simple flat-rate tariff

The minimal tariff has a single version with one `consumption` formula. The bare formula dict is shorthand for `{total: <formula>}`.

In [2]:
display_yaml("../examples/tariffs/simple.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 120.0

```

In [3]:
tariff = Tariff.from_yaml("../examples/tariffs/simple.yml")

# get_values returns the price series (€/MWh) — independent of meter data
tariff.get_values(start, end, output_resolution=res).head()

,timestamp,total
0,2025-12-01 00:00:00+00:00,120.0
1,2025-12-01 01:00:00+00:00,120.0
2,2025-12-01 02:00:00+00:00,120.0
3,2025-12-01 03:00:00+00:00,120.0
4,2025-12-01 04:00:00+00:00,120.0


## 2. Versioned tariffs

A tariff can have multiple versions, each starting from a `start` date. When you call `get_values` or `apply` with a range that spans version boundaries, the correct version is used for each segment automatically.

In [4]:
display_yaml("../examples/tariffs/dynamic.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 10.0
    variable_costs:
      - index: spot
        scalar: 1.05
- start: 2026-01-01T00:00:00+01:00
  consumption:
    constant_cost: 12.0
    variable_costs:
      - index: spot
        scalar: 1.10
```

Load the tariff and query December 2025 (version 1) and January 2026 (version 2):

In [5]:
tariff = Tariff.from_yaml("../examples/tariffs/dynamic.yml")

# December 2025 — version 1 applies (10 + spot * 1.05)
tariff.get_values(start, end, output_resolution=res).head()

,timestamp,total
0,2025-12-01 00:00:00+00:00,115.0
1,2025-12-01 01:00:00+00:00,115.0
2,2025-12-01 02:00:00+00:00,115.0
3,2025-12-01 03:00:00+00:00,115.0
4,2025-12-01 04:00:00+00:00,115.0


In [6]:
# January 2026 — version 2 applies (12 + spot * 1.10)
jan_start = dt.datetime(2026, 1, 1, tzinfo=dt.UTC)
jan_end   = dt.datetime(2026, 2, 1, tzinfo=dt.UTC)
tariff.get_values(jan_start, jan_end, output_resolution=res).head()

,timestamp,total
0,2026-01-01 00:00:00+00:00,144.0
1,2026-01-01 01:00:00+00:00,144.0
2,2026-01-01 02:00:00+00:00,144.0
3,2026-01-01 03:00:00+00:00,144.0
4,2026-01-01 04:00:00+00:00,144.0


## 3. Multiple cost groups

A single tariff version can define formulas for several cost groups at once. Each group is queried by passing `cost_group` to `get_values`.

In [7]:
display_yaml("../examples/tariffs/injection.yml")

```yaml
# Separate injection and consumption formulas
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 10.0
    variable_costs:
      - index: spot
        scalar: 1.05
  injection:
    variable_costs:
      - index: spot
        scalar: 1.0
```

In [8]:
tariff = Tariff.from_yaml("../examples/tariffs/injection.yml")

# Consumption price (default cost group)
tariff.get_values(start, end, output_resolution=res).head()

,timestamp,total
0,2025-12-01 00:00:00+00:00,115.0
1,2025-12-01 01:00:00+00:00,115.0
2,2025-12-01 02:00:00+00:00,115.0
3,2025-12-01 03:00:00+00:00,115.0
4,2025-12-01 04:00:00+00:00,115.0


Injection price — the same tariff, different cost group:

In [9]:
tariff.get_values(start, end, output_resolution=res, cost_group=CostGroup.INJECTION).head()

,timestamp,total
0,2025-12-01 00:00:00+00:00,100.0
1,2025-12-01 01:00:00+00:00,100.0
2,2025-12-01 02:00:00+00:00,100.0
3,2025-12-01 03:00:00+00:00,100.0
4,2025-12-01 04:00:00+00:00,100.0


In [10]:
display_yaml("../examples/tariffs/cost_types.yml")

```yaml
# Multiple named cost types within the consumption group
- start: 2025-01-01T00:00:00+01:00
  consumption:
    energy:
      constant_cost: 10.0
      variable_costs:
        - index: spot
          scalar: 1.05
    renewable_certificates:
      constant_cost: 5.0
    grid_fees:
      constant_cost: 8.0
```

## 4. Named cost types

A cost group can hold multiple **named** formulas. `get_values` then returns one column per name, plus a `total` column that sums them.

In [11]:
tariff = Tariff.from_yaml("../examples/tariffs/cost_types.yml")

# Returns one column per named formula + total
tariff.get_values(start, end, output_resolution=res).head()

,timestamp,energy,renewable_certificates,grid_fees,total
0,2025-12-01 00:00:00+00:00,115.0,5.0,8.0,128.0
1,2025-12-01 01:00:00+00:00,115.0,5.0,8.0,128.0
2,2025-12-01 02:00:00+00:00,115.0,5.0,8.0,128.0
3,2025-12-01 03:00:00+00:00,115.0,5.0,8.0,128.0
4,2025-12-01 04:00:00+00:00,115.0,5.0,8.0,128.0


## 5. Time-of-use (scheduled)

A `ScheduledFormulas` formula selects a different price depending on the time of day and day of the week. Scheduling can be used in any cost group; here it is in `consumption`.

In [12]:
display_yaml("../examples/tariffs/scheduled.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    schedule:
      - when: # formula valid for weekday mornings and weekend daytime
          - days: [monday, tuesday, wednesday, thursday, friday]
            start: "06:00:00"
            end: "10:00:00"
          - days: [saturday, sunday]
            start: "07:00:00"
            end: "19:00:00"
        formula:
          constant_cost: 300.0
      - when: # formula valid any day during late morning and evening (the first matching formula takes precedence, so this will not apply during the weekday morning and weekend daytime windows above)
          - start: "10:00:00"
            end: "13:00:00"
          - start: "18:00:00"
            end: "22:00:00"
        formula:
          constant_cost: 150.0
      - formula:
          constant_cost: 100.0 # formula valid at all times not covered by the previous definitions

```

Weekday prices at hourly resolution — the three price levels are visible across the day:

In [13]:
tariff = Tariff.from_yaml("../examples/tariffs/scheduled.yml")

# Monday 2025-12-01
tariff.get_values(start, dt.datetime(2025, 12, 2, tzinfo=dt.UTC), output_resolution=res)

,timestamp,total
0,2025-12-01 00:00:00+00:00,100.0
1,2025-12-01 01:00:00+00:00,100.0
2,2025-12-01 02:00:00+00:00,100.0
3,2025-12-01 03:00:00+00:00,100.0
4,2025-12-01 04:00:00+00:00,100.0
5,2025-12-01 05:00:00+00:00,100.0
6,2025-12-01 06:00:00+00:00,300.0
7,2025-12-01 07:00:00+00:00,300.0
8,2025-12-01 08:00:00+00:00,300.0
9,2025-12-01 09:00:00+00:00,300.0


On a Saturday only the weekend daytime slot (07:00–19:00) and the evening peak apply:

In [14]:
# Saturday 2025-12-06
sat_start = dt.datetime(2025, 12, 6, tzinfo=dt.UTC)
sat_end   = dt.datetime(2025, 12, 7, tzinfo=dt.UTC)
tariff.get_values(sat_start, sat_end, output_resolution=res)

,timestamp,total
0,2025-12-06 00:00:00+00:00,100.0
1,2025-12-06 01:00:00+00:00,100.0
2,2025-12-06 02:00:00+00:00,100.0
3,2025-12-06 03:00:00+00:00,100.0
4,2025-12-06 04:00:00+00:00,100.0
5,2025-12-06 05:00:00+00:00,100.0
6,2025-12-06 06:00:00+00:00,100.0
7,2025-12-06 07:00:00+00:00,300.0
8,2025-12-06 08:00:00+00:00,300.0
9,2025-12-06 09:00:00+00:00,300.0


## 6. Calculating actual cost with `apply()`

`get_values` returns a price series (€/MWh). To calculate the actual **cost** from meter readings, use `apply(consumption, injection)`.

`apply()` always requires both meters — pass an empty meter if a direction is unused. The result has **multi-index columns** — one level for the cost group, one for the cost name. A `(total, total)` column sums all groups.

In [15]:
display_yaml("../examples/tariffs/fixed.yml")

```yaml
- start: 2024-01-01T00:00:00+01:00
  consumption:
    constant_cost: 100.0
  injection:
    constant_cost: -20.0
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 90.0
  injection:
    constant_cost: -15.0
  fixed:
    period: P1M
    constant_cost: 10
- start: 2026-01-01T00:00:00+01:00
  consumption:
    constant_cost: 120.0
  injection:
    constant_cost: -25.0
  fixed:
    period: P1M
    constant_cost: 15
```

Load and apply — the result is grouped by cost group and aggregated to monthly buckets by default:

In [16]:
tariff = Tariff.from_yaml("../examples/tariffs/fixed.yml")

result = tariff.apply(consumption, injection, start, end)
result

,timestamp,consumption,fixed,injection,total
,,total,total,total,total
0,2025-12-01 00:00:00+00:00,66990.0,25.0,-4468.0,62547.0


In [17]:
# Extract a single cost group from the multi-index result
result["consumption"]

,total
0,66990.0
